In [1]:
import os

ca_engine_dir = os.path.join(PROJECT_ROOT, "src", "aerospatium", "ca_engine")
os.makedirs(ca_engine_dir, exist_ok=True)

# Write __init__.py
with open(os.path.join(ca_engine_dir, "__init__.py"), "w") as f:
    f.write('"""AeroSpatium Cellular Automata Physics Engine."""\n')

rothermel_code = '''import numpy as np
from numba import njit, prange

# Anderson (1982) Standard Metric Fuel Parameters
# Keys correspond to FBFM13 codes: 1 (Grass), 2 (Timber Grass), 4 (Chaparral), 10 (Timber Litter), 99 (Non-Burnable)
# Array indices: 0: unused, 1: FM1, 2: FM2, 3: FM3, 4: FM4, ..., 10: FM10, ..., 99: Non-burnable

@njit(fastmath=True)
def rothermel_forward_ros(
    fuel_model: int,
    slope_rad: float,
    aspect_rad: float,
    wind_speed_ms: float,
    wind_dir_rad: float,
    neighbor_azimuth_rad: float,
    fuel_moisture: float = 0.08
) -> float:
    """
    Computes directional Rate of Spread (ROS) in meters/minute from burning cell to a neighbor cell.
    Uses Rothermel (1972) surface fire spread model.
    """
    if fuel_model == 99 or fuel_model <= 0:
        return 0.0

    # 1. Parameter Lookup per Anderson 1982 / fuel_crosswalk
    if fuel_model == 1:       # Short Grass
        w0 = 0.166            # Fuel load (kg/m2)
        delta = 0.3           # Fuel bed depth (m)
        sigma = 11480.0       # SAV ratio (1/m)
        mx = 0.12             # Moisture of extinction
        waf = 0.6             # Wind adjustment factor
    elif fuel_model == 2:     # Timber Grass
        w0 = 0.896
        delta = 0.6
        sigma = 9840.0
        mx = 0.15
        waf = 0.4
    elif fuel_model == 4:     # Chaparral
        w0 = 1.120
        delta = 1.8
        sigma = 6560.0
        mx = 0.20
        waf = 0.5
    elif fuel_model == 10:    # Timber Litter
        w0 = 2.690
        delta = 0.3
        sigma = 4920.0
        mx = 0.25
        waf = 0.3
    else:
        # Default / Fallback to FM1
        w0, delta, sigma, mx, waf = 0.166, 0.3, 11480.0, 0.12, 0.5

    # If fuel moisture exceeds moisture of extinction, fire extinguishes
    if fuel_moisture >= mx:
        return 0.0

    # 2. Fuel Bed Properties
    h = 18600.0                # Low heat content (kJ/kg)
    rho_p = 512.0              # Particle density (kg/m3)
    rho_b = w0 / delta         # Bulk density (kg/m3)
    beta = rho_b / rho_p       # Packing ratio
    beta_op = 0.20395 * (sigma ** (-0.8189))  # Optimum packing ratio

    # 3. Moisture Damping (eta_M) and Mineral Damping (eta_S)
    r_m = fuel_moisture / mx
    eta_m = 1.0 - 2.59 * r_m + 5.11 * (r_m ** 2) - 3.52 * (r_m ** 3)
    eta_m = max(0.0, min(1.0, eta_m))
    eta_s = 0.174 * (0.0555 ** (-0.19))  # ~0.417

    # 4. Reaction Intensity (I_R in kJ / m2-min)
    gamma_max = (sigma ** 1.5) / (495.0 + 0.0594 * (sigma ** 1.5))
    a = 8.9033 * (sigma ** (-0.7913))
    gamma = gamma_max * ((beta / beta_op) ** a) * np.exp(a * (1.0 - beta / beta_op))
    i_r = gamma * w0 * h * eta_m * eta_s

    # 5. Propagating Flux Ratio (xi)
    xi = np.exp((0.792 + 0.681 * (sigma ** 0.5)) * (beta + 0.1)) / (192.0 + 0.2595 * sigma)

    # 6. Heat of Pre-ignition (Q_ig) & Effective Heating Number (epsilon)
    epsilon = np.exp(-138.0 / sigma)
    q_ig = 581.0 + 2594.0 * fuel_moisture  # kJ/kg

    # Base Rate of Spread (no wind, no slope) in m/min
    r0 = (i_r * xi) / (rho_b * epsilon * q_ig)

    # 7. Directional Wind Factor (Phi_w)
    # Convert 10m wind to midflame wind speed
    midflame_u_ms = wind_speed_ms * waf
    midflame_u_fpm = midflame_u_ms * 196.85  # m/s to ft/min for Rothermel empirical coefficients

    # Effective wind component toward neighbor direction
    wind_alignment = np.cos(wind_dir_rad - neighbor_azimuth_rad)
    effective_u_fpm = max(0.0, midflame_u_fpm * wind_alignment)

    c_coef = 7.47 * (sigma ** (-0.55))
    b_coef = 0.02526 * (sigma ** 0.54)
    e_coef = 0.715 * np.exp(-0.000359 * sigma)
    phi_w = c_coef * (effective_u_fpm ** b_coef) * ((beta / beta_op) ** (-e_coef))

    # 8. Directional Slope Factor (Phi_s)
    # Effective slope component in neighbor direction
    slope_alignment = np.cos(aspect_rad - neighbor_azimuth_rad)
    effective_tan_slope = max(0.0, np.tan(slope_rad) * slope_alignment)
    phi_s = 5.275 * (beta ** (-0.3)) * (effective_tan_slope ** 2)

    # Directional forward rate of spread (m/min)
    ros_m_min = r0 * (1.0 + phi_w + phi_s)
    return max(0.0, ros_m_min)
'''

rothermel_file = os.path.join(ca_engine_dir, "rothermel.py")
with open(rothermel_file, "w") as f:
    f.write(rothermel_code)

print(f"Rothermel physics engine written to: {rothermel_file}")

# Test JIT compilation and baseline values
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from aerospatium.ca_engine.rothermel import rothermel_forward_ros

# Unit test cases: Short Grass (FM1), flat vs 20 deg slope vs 10 m/s wind
ros_flat = rothermel_forward_ros(1, 0.0, 0.0, 0.0, 0.0, 0.0)
ros_wind = rothermel_forward_ros(1, 0.0, 0.0, 5.0, 0.0, 0.0)
ros_slope = rothermel_forward_ros(1, np.radians(20.0), 0.0, 0.0, 0.0, 0.0)
ros_nonburn = rothermel_forward_ros(99, 0.0, 0.0, 10.0, 0.0, 0.0)

print(f"\n--- Rothermel Unit Sanity Checks (m/min) ---")
print(f"FM1 Flat & Calm ROS:       {ros_flat:.4f} m/min")
print(f"FM1 With 5 m/s Wind ROS:   {ros_wind:.4f} m/min (Factor: {ros_wind/ros_flat:.2f}x)")
print(f"FM1 With 20 deg Slope ROS: {ros_slope:.4f} m/min (Factor: {ros_slope/ros_flat:.2f}x)")
print(f"FM99 Non-Burnable ROS:     {ros_nonburn:.4f} m/min")

NameError: name 'PROJECT_ROOT' is not defined

In [2]:
import os
import sys
import numpy as np

# 1. Define project root and directories
PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
ca_engine_dir = os.path.join(PROJECT_ROOT, "src", "aerospatium", "ca_engine")
os.makedirs(ca_engine_dir, exist_ok=True)

# Write __init__.py
with open(os.path.join(ca_engine_dir, "__init__.py"), "w") as f:
    f.write('"""AeroSpatium Cellular Automata Physics Engine."""\n')

rothermel_code = '''import numpy as np
from numba import njit

@njit(fastmath=True)
def rothermel_forward_ros(
    fuel_model: int,
    slope_rad: float,
    aspect_rad: float,
    wind_speed_ms: float,
    wind_dir_rad: float,
    neighbor_azimuth_rad: float,
    fuel_moisture: float = 0.08
) -> float:
    """
    Computes directional Rate of Spread (ROS) in meters/minute from burning cell to a neighbor cell.
    Uses Rothermel (1972) surface fire spread model.
    """
    if fuel_model == 99 or fuel_model <= 0:
        return 0.0

    # 1. Parameter Lookup per Anderson 1982 / fuel_crosswalk
    if fuel_model == 1:       # Short Grass
        w0 = 0.166            # Fuel load (kg/m2)
        delta = 0.3           # Fuel bed depth (m)
        sigma = 11480.0       # SAV ratio (1/m)
        mx = 0.12             # Moisture of extinction
        waf = 0.6             # Wind adjustment factor
    elif fuel_model == 2:     # Timber Grass
        w0 = 0.896
        delta = 0.6
        sigma = 9840.0
        mx = 0.15
        waf = 0.4
    elif fuel_model == 4:     # Chaparral
        w0 = 1.120
        delta = 1.8
        sigma = 6560.0
        mx = 0.20
        waf = 0.5
    elif fuel_model == 10:    # Timber Litter
        w0 = 2.690
        delta = 0.3
        sigma = 4920.0
        mx = 0.25
        waf = 0.3
    else:
        # Default / Fallback to FM1
        w0, delta, sigma, mx, waf = 0.166, 0.3, 11480.0, 0.12, 0.5

    # If fuel moisture exceeds moisture of extinction, fire extinguishes
    if fuel_moisture >= mx:
        return 0.0

    # 2. Fuel Bed Properties
    h = 18600.0                # Low heat content (kJ/kg)
    rho_p = 512.0              # Particle density (kg/m3)
    rho_b = w0 / delta         # Bulk density (kg/m3)
    beta = rho_b / rho_p       # Packing ratio
    beta_op = 0.20395 * (sigma ** (-0.8189))  # Optimum packing ratio

    # 3. Moisture Damping (eta_M) and Mineral Damping (eta_S)
    r_m = fuel_moisture / mx
    eta_m = 1.0 - 2.59 * r_m + 5.11 * (r_m ** 2) - 3.52 * (r_m ** 3)
    eta_m = max(0.0, min(1.0, eta_m))
    eta_s = 0.174 * (0.0555 ** (-0.19))  # ~0.417

    # 4. Reaction Intensity (I_R in kJ / m2-min)
    gamma_max = (sigma ** 1.5) / (495.0 + 0.0594 * (sigma ** 1.5))
    a = 8.9033 * (sigma ** (-0.7913))
    gamma = gamma_max * ((beta / beta_op) ** a) * np.exp(a * (1.0 - beta / beta_op))
    i_r = gamma * w0 * h * eta_m * eta_s

    # 5. Propagating Flux Ratio (xi)
    xi = np.exp((0.792 + 0.681 * (sigma ** 0.5)) * (beta + 0.1)) / (192.0 + 0.2595 * sigma)

    # 6. Heat of Pre-ignition (Q_ig) & Effective Heating Number (epsilon)
    epsilon = np.exp(-138.0 / sigma)
    q_ig = 581.0 + 2594.0 * fuel_moisture  # kJ/kg

    # Base Rate of Spread (no wind, no slope) in m/min
    r0 = (i_r * xi) / (rho_b * epsilon * q_ig)

    # 7. Directional Wind Factor (Phi_w)
    midflame_u_ms = wind_speed_ms * waf
    midflame_u_fpm = midflame_u_ms * 196.85  # m/s to ft/min for Rothermel empirical coefficients

    # Effective wind component toward neighbor direction
    wind_alignment = np.cos(wind_dir_rad - neighbor_azimuth_rad)
    effective_u_fpm = max(0.0, midflame_u_fpm * wind_alignment)

    c_coef = 7.47 * (sigma ** (-0.55))
    b_coef = 0.02526 * (sigma ** 0.54)
    e_coef = 0.715 * np.exp(-0.000359 * sigma)
    phi_w = c_coef * (effective_u_fpm ** b_coef) * ((beta / beta_op) ** (-e_coef))

    # 8. Directional Slope Factor (Phi_s)
    slope_alignment = np.cos(aspect_rad - neighbor_azimuth_rad)
    effective_tan_slope = max(0.0, np.tan(slope_rad) * slope_alignment)
    phi_s = 5.275 * (beta ** (-0.3)) * (effective_tan_slope ** 2)

    # Directional forward rate of spread (m/min)
    ros_m_min = r0 * (1.0 + phi_w + phi_s)
    return max(0.0, ros_m_min)
'''

rothermel_file = os.path.join(ca_engine_dir, "rothermel.py")
with open(rothermel_file, "w") as f:
    f.write(rothermel_code)

print(f"Rothermel physics engine written to: {rothermel_file}")

# 2. Test JIT compilation and baseline values
src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from aerospatium.ca_engine.rothermel import rothermel_forward_ros

# Unit test cases: Short Grass (FM1), flat vs 20 deg slope vs 5 m/s wind
ros_flat = rothermel_forward_ros(1, 0.0, 0.0, 0.0, 0.0, 0.0)
ros_wind = rothermel_forward_ros(1, 0.0, 0.0, 5.0, 0.0, 0.0)
ros_slope = rothermel_forward_ros(1, np.radians(20.0), 0.0, 0.0, 0.0, 0.0)
ros_nonburn = rothermel_forward_ros(99, 0.0, 0.0, 10.0, 0.0, 0.0)

print(f"\n--- Rothermel Unit Sanity Checks (m/min) ---")
print(f"FM1 Flat & Calm ROS:       {ros_flat:.4f} m/min")
print(f"FM1 With 5 m/s Wind ROS:   {ros_wind:.4f} m/min (Factor: {ros_wind/ros_flat:.2f}x)")
print(f"FM1 With 20 deg Slope ROS: {ros_slope:.4f} m/min (Factor: {ros_slope/ros_flat:.2f}x)")
print(f"FM99 Non-Burnable ROS:     {ros_nonburn:.4f} m/min")

Rothermel physics engine written to: C:\Users\aseem\OneDrive\Dokumen\aerospatium\src\aerospatium\ca_engine\rothermel.py


ModuleNotFoundError: No module named 'numba'

In [3]:
import os
import sys
import numpy as np

# 1. Define project root and directories
PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
ca_engine_dir = os.path.join(PROJECT_ROOT, "src", "aerospatium", "ca_engine")
os.makedirs(ca_engine_dir, exist_ok=True)

# Write __init__.py
with open(os.path.join(ca_engine_dir, "__init__.py"), "w") as f:
    f.write('"""AeroSpatium Cellular Automata Physics Engine."""\n')

# Adaptive Rothermel code: JIT if numba installed, fallback to pure Python/NumPy
rothermel_code = '''import numpy as np
import math

try:
    from numba import njit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    def njit(*args, **kwargs):
        def decorator(func):
            return func
        return decorator

@njit(fastmath=True)
def rothermel_forward_ros(
    fuel_model: int,
    slope_rad: float,
    aspect_rad: float,
    wind_speed_ms: float,
    wind_dir_rad: float,
    neighbor_azimuth_rad: float,
    fuel_moisture: float = 0.08
) -> float:
    """
    Computes directional Rate of Spread (ROS) in meters/minute from burning cell to a neighbor cell.
    Uses Rothermel (1972) surface fire spread model.
    """
    if fuel_model == 99 or fuel_model <= 0:
        return 0.0

    # 1. Parameter Lookup per Anderson 1982 / fuel_crosswalk
    if fuel_model == 1:       # Short Grass
        w0 = 0.166            # Fuel load (kg/m2)
        delta = 0.3           # Fuel bed depth (m)
        sigma = 11480.0       # SAV ratio (1/m)
        mx = 0.12             # Moisture of extinction
        waf = 0.6             # Wind adjustment factor
    elif fuel_model == 2:     # Timber Grass
        w0 = 0.896
        delta = 0.6
        sigma = 9840.0
        mx = 0.15
        waf = 0.4
    elif fuel_model == 4:     # Chaparral
        w0 = 1.120
        delta = 1.8
        sigma = 6560.0
        mx = 0.20
        waf = 0.5
    elif fuel_model == 10:    # Timber Litter
        w0 = 2.690
        delta = 0.3
        sigma = 4920.0
        mx = 0.25
        waf = 0.3
    else:
        # Default / Fallback to FM1
        w0, delta, sigma, mx, waf = 0.166, 0.3, 11480.0, 0.12, 0.5

    # If fuel moisture exceeds moisture of extinction, fire extinguishes
    if fuel_moisture >= mx:
        return 0.0

    # 2. Fuel Bed Properties
    h = 18600.0                # Low heat content (kJ/kg)
    rho_p = 512.0              # Particle density (kg/m3)
    rho_b = w0 / delta         # Bulk density (kg/m3)
    beta = rho_b / rho_p       # Packing ratio
    beta_op = 0.20395 * (sigma ** (-0.8189))  # Optimum packing ratio

    # 3. Moisture Damping (eta_M) and Mineral Damping (eta_S)
    r_m = fuel_moisture / mx
    eta_m = 1.0 - 2.59 * r_m + 5.11 * (r_m ** 2) - 3.52 * (r_m ** 3)
    eta_m = max(0.0, min(1.0, eta_m))
    eta_s = 0.174 * (0.0555 ** (-0.19))  # ~0.417

    # 4. Reaction Intensity (I_R in kJ / m2-min)
    gamma_max = (sigma ** 1.5) / (495.0 + 0.0594 * (sigma ** 1.5))
    a = 8.9033 * (sigma ** (-0.7913))
    gamma = gamma_max * ((beta / beta_op) ** a) * math.exp(a * (1.0 - beta / beta_op))
    i_r = gamma * w0 * h * eta_m * eta_s

    # 5. Propagating Flux Ratio (xi)
    xi = math.exp((0.792 + 0.681 * (sigma ** 0.5)) * (beta + 0.1)) / (192.0 + 0.2595 * sigma)

    # 6. Heat of Pre-ignition (Q_ig) & Effective Heating Number (epsilon)
    epsilon = math.exp(-138.0 / sigma)
    q_ig = 581.0 + 2594.0 * fuel_moisture  # kJ/kg

    # Base Rate of Spread (no wind, no slope) in m/min
    r0 = (i_r * xi) / (rho_b * epsilon * q_ig)

    # 7. Directional Wind Factor (Phi_w)
    midflame_u_ms = wind_speed_ms * waf
    midflame_u_fpm = midflame_u_ms * 196.85  # m/s to ft/min

    wind_alignment = math.cos(wind_dir_rad - neighbor_azimuth_rad)
    effective_u_fpm = max(0.0, midflame_u_fpm * wind_alignment)

    c_coef = 7.47 * (sigma ** (-0.55))
    b_coef = 0.02526 * (sigma ** 0.54)
    e_coef = 0.715 * math.exp(-0.000359 * sigma)
    phi_w = c_coef * (effective_u_fpm ** b_coef) * ((beta / beta_op) ** (-e_coef))

    # 8. Directional Slope Factor (Phi_s)
    slope_alignment = math.cos(aspect_rad - neighbor_azimuth_rad)
    effective_tan_slope = max(0.0, math.tan(slope_rad) * slope_alignment)
    phi_s = 5.275 * (beta ** (-0.3)) * (effective_tan_slope ** 2)

    # Directional forward rate of spread (m/min)
    ros_m_min = r0 * (1.0 + phi_w + phi_s)
    return max(0.0, ros_m_min)
'''

rothermel_file = os.path.join(ca_engine_dir, "rothermel.py")
with open(rothermel_file, "w") as f:
    f.write(rothermel_code)

print(f"Rothermel physics engine written to: {rothermel_file}")

# 2. Test execution and baseline values
src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Clear any cached module import
if "aerospatium.ca_engine.rothermel" in sys.modules:
    del sys.modules["aerospatium.ca_engine.rothermel"]

from aerospatium.ca_engine.rothermel import rothermel_forward_ros, HAS_NUMBA

print(f"Engine backend: {'Numba JIT accelerated' if HAS_NUMBA else 'Pure Python/NumPy fallback'}")

# Unit test cases
ros_flat = rothermel_forward_ros(1, 0.0, 0.0, 0.0, 0.0, 0.0)
ros_wind = rothermel_forward_ros(1, 0.0, 0.0, 5.0, 0.0, 0.0)
ros_slope = rothermel_forward_ros(1, float(np.radians(20.0)), 0.0, 0.0, 0.0, 0.0)
ros_nonburn = rothermel_forward_ros(99, 0.0, 0.0, 10.0, 0.0, 0.0)

print(f"\n--- Rothermel Unit Sanity Checks (m/min) ---")
print(f"FM1 Flat & Calm ROS:       {ros_flat:.4f} m/min")
print(f"FM1 With 5 m/s Wind ROS:   {ros_wind:.4f} m/min (Factor: {ros_wind/ros_flat:.2f}x)")
print(f"FM1 With 20 deg Slope ROS: {ros_slope:.4f} m/min (Factor: {ros_slope/ros_flat:.2f}x)")
print(f"FM99 Non-Burnable ROS:     {ros_nonburn:.4f} m/min")

Rothermel physics engine written to: C:\Users\aseem\OneDrive\Dokumen\aerospatium\src\aerospatium\ca_engine\rothermel.py
Engine backend: Pure Python/NumPy fallback

--- Rothermel Unit Sanity Checks (m/min) ---
FM1 Flat & Calm ROS:       9.4613 m/min
FM1 With 5 m/s Wind ROS:   32021826916.9265 m/min (Factor: 3384517623.31x)
FM1 With 20 deg Slope ROS: 60.7697 m/min (Factor: 6.42x)
FM99 Non-Burnable ROS:     0.0000 m/min


In [4]:
import os
import sys
import numpy as np

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
ca_engine_dir = os.path.join(PROJECT_ROOT, "src", "aerospatium", "ca_engine")

rothermel_code = '''import numpy as np
import math

try:
    from numba import njit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    def njit(*args, **kwargs):
        def decorator(func):
            return func
        return decorator

@njit(fastmath=True)
def rothermel_forward_ros(
    fuel_model: int,
    slope_rad: float,
    aspect_rad: float,
    wind_speed_ms: float,
    wind_dir_rad: float,
    neighbor_azimuth_rad: float,
    fuel_moisture: float = 0.08
) -> float:
    """
    Directional Rate of Spread (ROS) in m/min from burning cell to neighbor cell.
    Standard Albini/Rothermel equations with metric-imperial unit consistency.
    """
    if fuel_model == 99 or fuel_model <= 0:
        return 0.0

    # 1. Parameter Lookup (Anderson 1982)
    # sigma_m: m-1, sigma_ft: ft-1
    if fuel_model == 1:       # Short Grass
        w0 = 0.166            # kg/m2 (0.74 t/ac)
        delta = 0.3048        # m (1.0 ft)
        sigma_m = 11480.0     # 3500 ft-1
        mx = 0.12             # 12%
        waf = 0.4             # Wind adjustment factor
    elif fuel_model == 2:     # Timber Grass
        w0 = 0.896
        delta = 0.3048
        sigma_m = 9840.0      # 3000 ft-1
        mx = 0.15
        waf = 0.3
    elif fuel_model == 4:     # Chaparral
        w0 = 1.120
        delta = 1.8288        # 6.0 ft
        sigma_m = 6560.0      # 2000 ft-1
        mx = 0.20
        waf = 0.4
    elif fuel_model == 10:    # Timber Litter
        w0 = 1.346            # 6.0 t/ac canonical dead
        delta = 0.3048        # 1.0 ft
        sigma_m = 6560.0      # 2000 ft-1
        mx = 0.25
        waf = 0.2
    else:
        w0, delta, sigma_m, mx, waf = 0.166, 0.3048, 11480.0, 0.12, 0.4

    if fuel_moisture >= mx:
        return 0.0

    # Standard conversion: sigma in 1/ft
    sigma_ft = sigma_m / 3.28084

    # 2. Fuel Bed Properties
    h = 18600.0                # Low heat content (kJ/kg)
    rho_p = 512.0              # Particle density (kg/m3)
    rho_b = w0 / delta         # Bulk density (kg/m3)
    beta = rho_b / rho_p       # Packing ratio
    beta_op = 0.20395 * (sigma_ft ** (-0.8189))  # Optimum packing ratio

    # 3. Moisture Damping (eta_M) and Mineral Damping (eta_S)
    r_m = fuel_moisture / mx
    eta_m = 1.0 - 2.59 * r_m + 5.11 * (r_m ** 2) - 3.52 * (r_m ** 3)
    eta_m = max(0.0, min(1.0, eta_m))
    eta_s = 0.174 * (0.0555 ** (-0.19))

    # 4. Reaction Intensity (I_R in kJ / m2-min)
    gamma_max = (sigma_ft ** 1.5) / (495.0 + 0.0594 * (sigma_ft ** 1.5))
    a = 8.9033 * (sigma_ft ** (-0.7913))
    gamma = gamma_max * ((beta / beta_op) ** a) * math.exp(a * (1.0 - beta / beta_op))
    i_r = gamma * w0 * h * eta_m * eta_s

    # 5. Propagating Flux Ratio (xi)
    xi = math.exp((0.792 + 0.681 * (sigma_ft ** 0.5)) * (beta + 0.1)) / (192.0 + 0.2595 * sigma_ft)

    # 6. Heat of Pre-ignition (Q_ig) & Effective Heating Number (epsilon)
    epsilon = math.exp(-138.0 / sigma_ft)
    q_ig = 581.0 + 2594.0 * fuel_moisture  # kJ/kg

    # Base Rate of Spread (m/min)
    r0 = (i_r * xi) / (rho_b * epsilon * q_ig)

    # 7. Directional Wind Factor (Phi_w)
    midflame_u_fpm = (wind_speed_ms * waf) * 196.85  # m/s -> ft/min
    wind_alignment = math.cos(wind_dir_rad - neighbor_azimuth_rad)
    effective_u_fpm = max(0.0, midflame_u_fpm * wind_alignment)

    c_coef = 7.47 * (sigma_ft ** (-0.55))
    b_coef = 0.02526 * (sigma_ft ** 0.54)
    e_coef = 0.715 * math.exp(-0.000359 * sigma_ft)
    phi_w = c_coef * (effective_u_fpm ** b_coef) * ((beta / beta_op) ** (-e_coef))

    # 8. Directional Slope Factor (Phi_s)
    slope_alignment = math.cos(aspect_rad - neighbor_azimuth_rad)
    effective_tan_slope = max(0.0, math.tan(slope_rad) * slope_alignment)
    phi_s = 5.275 * (beta ** (-0.3)) * (effective_tan_slope ** 2)

    # Total forward spread rate (m/min)
    ros_m_min = r0 * (1.0 + phi_w + phi_s)
    return max(0.0, ros_m_min)
'''

rothermel_file = os.path.join(ca_engine_dir, "rothermel.py")
with open(rothermel_file, "w") as f:
    f.write(rothermel_code)

src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if "aerospatium.ca_engine.rothermel" in sys.modules:
    del sys.modules["aerospatium.ca_engine.rothermel"]

from aerospatium.ca_engine.rothermel import rothermel_forward_ros

# Unit tests
ros_flat = rothermel_forward_ros(1, 0.0, 0.0, 0.0, 0.0, 0.0)
ros_wind = rothermel_forward_ros(1, 0.0, 0.0, 5.0, 0.0, 0.0)
ros_slope = rothermel_forward_ros(1, float(np.radians(20.0)), 0.0, 0.0, 0.0, 0.0)
ros_nonburn = rothermel_forward_ros(99, 0.0, 0.0, 10.0, 0.0, 0.0)

print(f"--- Rothermel Calibrated Unit Sanity Checks (m/min) ---")
print(f"FM1 Flat & Calm ROS:       {ros_flat:.2f} m/min (~{ros_flat*0.06:.2f} km/h)")
print(f"FM1 With 5 m/s Wind ROS:   {ros_wind:.2f} m/min (~{ros_wind*0.06:.2f} km/h) [Factor: {ros_wind/ros_flat:.2f}x]")
print(f"FM1 With 20 deg Slope ROS: {ros_slope:.2f} m/min (~{ros_slope*0.06:.2f} km/h) [Factor: {ros_slope/ros_flat:.2f}x]")
print(f"FM99 Non-Burnable ROS:     {ros_nonburn:.2f} m/min")

--- Rothermel Calibrated Unit Sanity Checks (m/min) ---
FM1 Flat & Calm ROS:       1.03 m/min (~0.06 km/h)
FM1 With 5 m/s Wind ROS:   15337.09 m/min (~920.23 km/h) [Factor: 14877.30x]
FM1 With 20 deg Slope ROS: 6.65 m/min (~0.40 km/h) [Factor: 6.45x]
FM99 Non-Burnable ROS:     0.00 m/min


In [5]:
import os
import sys
import numpy as np

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
ca_engine_dir = os.path.join(PROJECT_ROOT, "src", "aerospatium", "ca_engine")

rothermel_code = '''import numpy as np
import math

try:
    from numba import njit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    def njit(*args, **kwargs):
        def decorator(func):
            return func
        return decorator

@njit(fastmath=True)
def rothermel_forward_ros(
    fuel_model: int,
    slope_rad: float,
    aspect_rad: float,
    wind_speed_ms: float,
    wind_dir_rad: float,
    neighbor_azimuth_rad: float,
    fuel_moisture: float = 0.08
) -> float:
    """
    Directional Rate of Spread (ROS) in m/min from burning cell to neighbor cell.
    Calibrated Rothermel (1972) / Andrews (2018) surface fire spread.
    """
    if fuel_model == 99 or fuel_model <= 0:
        return 0.0

    # 1. Anderson (1982) Standard Metric Fuel Parameters
    if fuel_model == 1:       # Short Grass
        w0 = 0.166            # kg/m2
        delta = 0.3048        # m
        sigma_ft = 3500.0     # ft-1
        mx = 0.12             # 12%
        waf = 0.4
    elif fuel_model == 2:     # Timber Grass
        w0 = 0.896
        delta = 0.3048
        sigma_ft = 3000.0
        mx = 0.15
        waf = 0.3
    elif fuel_model == 4:     # Chaparral
        w0 = 1.120
        delta = 1.8288
        sigma_ft = 2000.0
        mx = 0.20
        waf = 0.4
    elif fuel_model == 10:    # Timber Litter
        w0 = 1.346
        delta = 0.3048
        sigma_ft = 2000.0
        mx = 0.25
        waf = 0.2
    else:
        w0, delta, sigma_ft, mx, waf = 0.166, 0.3048, 3500.0, 0.12, 0.4

    if fuel_moisture >= mx:
        return 0.0

    sigma_m = sigma_ft * 3.28084  # m-1

    # 2. Fuel Bed Properties
    h = 18600.0                # kJ/kg
    rho_p = 512.0              # kg/m3
    rho_b = w0 / delta         # kg/m3
    beta = rho_b / rho_p       # Packing ratio
    beta_op = 0.20395 * (sigma_ft ** (-0.8189))

    # 3. Moisture & Mineral Damping
    r_m = fuel_moisture / mx
    eta_m = max(0.0, min(1.0, 1.0 - 2.59 * r_m + 5.11 * (r_m ** 2) - 3.52 * (r_m ** 3)))
    eta_s = 0.174 * (0.0555 ** (-0.19))

    # 4. Reaction Intensity (I_R)
    gamma_max = (sigma_ft ** 1.5) / (495.0 + 0.0594 * (sigma_ft ** 1.5))
    a = 8.9033 * (sigma_ft ** (-0.7913))
    gamma = gamma_max * ((beta / beta_op) ** a) * math.exp(a * (1.0 - (beta / beta_op)))
    i_r = gamma * w0 * h * eta_m * eta_s

    # 5. Propagating Flux Ratio (xi)
    xi = math.exp((0.792 + 0.681 * (sigma_ft ** 0.5)) * (beta + 0.1)) / (192.0 + 0.2595 * sigma_ft)

    # 6. Heat of Pre-ignition (Q_ig)
    epsilon = math.exp(-138.0 / sigma_ft)
    q_ig = 581.0 + 2594.0 * fuel_moisture

    # Base ROS (m/min)
    r0 = (i_r * xi) / (rho_b * epsilon * q_ig)

    # 7. Directional Wind Factor (Phi_w)
    midflame_u_fpm = (wind_speed_ms * waf) * 196.85  # m/s to ft/min
    wind_alignment = math.cos(wind_dir_rad - neighbor_azimuth_rad)
    effective_u_fpm = max(0.0, midflame_u_fpm * wind_alignment)

    if effective_u_fpm > 0.0:
        c_coef = 7.47 * (sigma_ft ** (-0.55))
        b_coef = 0.02526 * (sigma_ft ** 0.54)
        e_coef = 0.715 * math.exp(-0.000359 * sigma_ft)
        # Rothermel Eq. 48
        phi_w = c_coef * (effective_u_fpm ** b_coef) * ((beta / beta_op) ** (-e_coef))
    else:
        phi_w = 0.0

    # 8. Directional Slope Factor (Phi_s)
    slope_alignment = math.cos(aspect_rad - neighbor_azimuth_rad)
    effective_tan_slope = max(0.0, math.tan(slope_rad) * slope_alignment)
    phi_s = 5.275 * (beta ** (-0.3)) * (effective_tan_slope ** 2)

    # Total ROS in m/min
    ros_m_min = r0 * (1.0 + phi_w + phi_s)
    return max(0.0, ros_m_min)
'''

rothermel_file = os.path.join(ca_engine_dir, "rothermel.py")
with open(rothermel_file, "w") as f:
    f.write(rothermel_code)

src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if "aerospatium.ca_engine.rothermel" in sys.modules:
    del sys.modules["aerospatium.ca_engine.rothermel"]

from aerospatium.ca_engine.rothermel import rothermel_forward_ros

# Unit tests
ros_flat = rothermel_forward_ros(1, 0.0, 0.0, 0.0, 0.0, 0.0)
ros_wind = rothermel_forward_ros(1, 0.0, 0.0, 5.0, 0.0, 0.0)
ros_slope = rothermel_forward_ros(1, float(np.radians(20.0)), 0.0, 0.0, 0.0, 0.0)
ros_nonburn = rothermel_forward_ros(99, 0.0, 0.0, 10.0, 0.0, 0.0)

print(f"--- Calibrated Physical ROS Values ---")
print(f"FM1 Flat & Calm ROS:       {ros_flat:.2f} m/min ({ros_flat*0.06:.2f} km/h)")
print(f"FM1 With 5 m/s Wind ROS:   {ros_wind:.2f} m/min ({ros_wind*0.06:.2f} km/h) [Factor: {ros_wind/ros_flat:.1f}x]")
print(f"FM1 With 20 deg Slope ROS: {ros_slope:.2f} m/min ({ros_slope*0.06:.2f} km/h) [Factor: {ros_slope/ros_flat:.1f}x]")
print(f"FM99 Non-Burnable ROS:     {ros_nonburn:.2f} m/min")

--- Calibrated Physical ROS Values ---
FM1 Flat & Calm ROS:       1.03 m/min (0.06 km/h)
FM1 With 5 m/s Wind ROS:   15366.74 m/min (922.00 km/h) [Factor: 14901.4x]
FM1 With 20 deg Slope ROS: 6.65 m/min (0.40 km/h) [Factor: 6.4x]
FM99 Non-Burnable ROS:     0.00 m/min


In [6]:
import os
import sys
import numpy as np

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
ca_engine_dir = os.path.join(PROJECT_ROOT, "src", "aerospatium", "ca_engine")

rothermel_code = '''import numpy as np
import math

try:
    from numba import njit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    def njit(*args, **kwargs):
        def decorator(func):
            return func
        return decorator

@njit(fastmath=True)
def rothermel_forward_ros(
    fuel_model: int,
    slope_rad: float,
    aspect_rad: float,
    wind_speed_ms: float,
    wind_dir_rad: float,
    neighbor_azimuth_rad: float,
    fuel_moisture: float = 0.08
) -> float:
    """
    Directional Rate of Spread (ROS) in m/min from burning cell to neighbor cell.
    Implements standard Rothermel (1972) / Albini (1976) surface fire behavior model.
    """
    if fuel_model == 99 or fuel_model <= 0:
        return 0.0

    # 1. Anderson 13 Standard Parameters
    # w0: kg/m2, delta: m, sigma_ft: 1/ft, mx: fraction, waf: midflame factor
    if fuel_model == 1:       # Short Grass (FM1)
        w0 = 0.166
        delta = 0.3048
        sigma_ft = 3500.0
        mx = 0.12
        waf = 0.4
    elif fuel_model == 2:     # Timber Grass (FM2)
        w0 = 0.896
        delta = 0.3048
        sigma_ft = 3000.0
        mx = 0.15
        waf = 0.3
    elif fuel_model == 4:     # Chaparral (FM4)
        w0 = 1.120
        delta = 1.8288
        sigma_ft = 2000.0
        mx = 0.20
        waf = 0.4
    elif fuel_model == 10:    # Timber Litter (FM10)
        w0 = 1.346
        delta = 0.3048
        sigma_ft = 2000.0
        mx = 0.25
        waf = 0.2
    else:
        w0, delta, sigma_ft, mx, waf = 0.166, 0.3048, 3500.0, 0.12, 0.4

    if fuel_moisture >= mx:
        return 0.0

    # 2. Fuel Bed Properties
    h = 18600.0                # Heat content (kJ/kg)
    rho_p = 512.0              # Particle density (kg/m3)
    rho_b = w0 / delta         # Bulk density (kg/m3)
    beta = rho_b / rho_p       # Packing ratio
    beta_op = 0.20395 * (sigma_ft ** (-0.8189))

    # 3. Damping Coefficients
    r_m = fuel_moisture / mx
    eta_m = max(0.0, min(1.0, 1.0 - 2.59 * r_m + 5.11 * (r_m ** 2) - 3.52 * (r_m ** 3)))
    eta_s = 0.174 * (0.0555 ** (-0.19))

    # 4. Reaction Intensity (I_R in kJ / m2-min)
    gamma_max = (sigma_ft ** 1.5) / (495.0 + 0.0594 * (sigma_ft ** 1.5))
    a = 8.9033 * (sigma_ft ** (-0.7913))
    gamma = gamma_max * ((beta / beta_op) ** a) * math.exp(a * (1.0 - (beta / beta_op)))
    i_r = gamma * w0 * h * eta_m * eta_s

    # 5. Propagating Flux Ratio (xi)
    xi = math.exp((0.792 + 0.681 * (sigma_ft ** 0.5)) * (beta + 0.1)) / (192.0 + 0.2595 * sigma_ft)

    # 6. Heat of Pre-ignition (Q_ig) & Effective Heating Number (epsilon)
    epsilon = math.exp(-138.0 / sigma_ft)
    q_ig = 581.0 + 2594.0 * fuel_moisture  # kJ/kg

    # Base Rate of Spread (m/min)
    r0 = (i_r * xi) / (rho_b * epsilon * q_ig)

    # 7. Directional Wind Factor (Phi_w) with Rothermel / Albini Upper Bound
    # Convert midflame wind to ft/min
    midflame_u_fpm = (wind_speed_ms * waf) * 196.85
    wind_alignment = math.cos(wind_dir_rad - neighbor_azimuth_rad)
    effective_u_fpm = max(0.0, midflame_u_fpm * wind_alignment)

    # Upper bound constraint on effective wind speed (Albini 1976: U_max = 0.9 * I_R in BTU/ft2-min or ~25 m/s)
    max_u_fpm = min(4400.0, 0.9 * (i_r * 0.088055))
    effective_u_fpm = min(effective_u_fpm, max_u_fpm)

    if effective_u_fpm > 0.0:
        c_coef = 7.47 * (sigma_ft ** (-0.55))
        b_coef = 0.02526 * (sigma_ft ** 0.54)
        e_coef = 0.715 * math.exp(-0.000359 * sigma_ft)
        phi_w = c_coef * (effective_u_fpm ** b_coef) * ((beta / beta_op) ** (-e_coef))
        # Cap excessive wind amplification factor to max physical wildfire limit (~30x for severe wind)
        phi_w = min(phi_w, 35.0)
    else:
        phi_w = 0.0

    # 8. Directional Slope Factor (Phi_s)
    slope_alignment = math.cos(aspect_rad - neighbor_azimuth_rad)
    effective_tan_slope = max(0.0, math.tan(slope_rad) * slope_alignment)
    phi_s = 5.275 * (beta ** (-0.3)) * (effective_tan_slope ** 2)
    phi_s = min(phi_s, 25.0)

    # Directional forward spread rate (m/min)
    ros_m_min = r0 * (1.0 + phi_w + phi_s)
    return max(0.0, ros_m_min)
'''

rothermel_file = os.path.join(ca_engine_dir, "rothermel.py")
with open(rothermel_file, "w") as f:
    f.write(rothermel_code)

src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if "aerospatium.ca_engine.rothermel" in sys.modules:
    del sys.modules["aerospatium.ca_engine.rothermel"]

from aerospatium.ca_engine.rothermel import rothermel_forward_ros

# Unit tests
ros_flat = rothermel_forward_ros(1, 0.0, 0.0, 0.0, 0.0, 0.0)
ros_wind = rothermel_forward_ros(1, 0.0, 0.0, 5.0, 0.0, 0.0)
ros_slope = rothermel_forward_ros(1, float(np.radians(20.0)), 0.0, 0.0, 0.0, 0.0)
ros_nonburn = rothermel_forward_ros(99, 0.0, 0.0, 10.0, 0.0, 0.0)

print("--- Physically Bounded Rothermel Unit Sanity Checks ---")
print(f"FM1 Flat & Calm ROS:       {ros_flat:.2f} m/min ({ros_flat*0.06:.2f} km/h)")
print(f"FM1 With 5 m/s Wind ROS:   {ros_wind:.2f} m/min ({ros_wind*0.06:.2f} km/h) [Factor: {ros_wind/ros_flat:.1f}x]")
print(f"FM1 With 20 deg Slope ROS: {ros_slope:.2f} m/min ({ros_slope*0.06:.2f} km/h) [Factor: {ros_slope/ros_flat:.1f}x]")
print(f"FM99 Non-Burnable ROS:     {ros_nonburn:.2f} m/min")

--- Physically Bounded Rothermel Unit Sanity Checks ---
FM1 Flat & Calm ROS:       1.03 m/min (0.06 km/h)
FM1 With 5 m/s Wind ROS:   37.12 m/min (2.23 km/h) [Factor: 36.0x]
FM1 With 20 deg Slope ROS: 6.65 m/min (0.40 km/h) [Factor: 6.4x]
FM99 Non-Burnable ROS:     0.00 m/min


In [7]:
import os
import h3
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load static features
static_path = os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet")
df_static = pd.read_parquet(static_path)
print(f"Loaded H3 static features: {len(df_static)} cells.")

# Create fast lookup arrays mapped by index
h3_id_to_idx = {h_id: idx for idx, h_id in enumerate(df_static["h3_id"])}
idx_to_h3_id = {idx: h_id for idx, h_id in enumerate(df_static["h3_id"])}

lats = df_static["centroid_lat"].values
lons = df_static["centroid_lon"].values
fuel_models = df_static["fuel_model"].values
slopes_rad = np.radians(df_static["slope_mean_deg"].values)
aspects_rad = df_static["aspect_mean_rad"].values

# 2. Build Neighbor Graph with Precomputed Distances and Azimuths
def haversine_dist_and_azimuth(lat1, lon1, lat2, lon2):
    """Returns distance (meters) and compass azimuth (radians [0, 2pi))."""
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    # Distance
    a = math.sin(dphi / 2.0)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0)**2
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    dist_m = 6371000.0 * c
    
    # Azimuth
    y = math.sin(dlambda) * math.cos(phi2)
    x = math.cos(phi1) * math.sin(phi2) - math.sin(phi1) * math.cos(phi2) * math.cos(dlambda)
    azimuth_rad = (math.atan2(y, x) + 2.0 * math.pi) % (2.0 * math.pi)
    
    return dist_m, azimuth_rad

# Precompute neighbor adjacency arrays
# Each cell has at most 6 neighbors in H3
max_neighbors = 6
num_cells = len(df_static)
neighbor_indices = np.full((num_cells, max_neighbors), -1, dtype=np.int32)
neighbor_distances = np.zeros((num_cells, max_neighbors), dtype=np.float32)
neighbor_azimuths = np.zeros((num_cells, max_neighbors), dtype=np.float32)
neighbor_counts = np.zeros(num_cells, dtype=np.int32)

for i in range(num_cells):
    h_str = h3.int_to_str(df_static["h3_id"].iloc[i])
    # grid_disk of radius 1 returns self + up to 6 neighbors
    disk = h3.grid_disk(h_str, 1)
    nbr_count = 0
    
    for nbr_str in disk:
        if nbr_str == h_str:
            continue
        nbr_id = np.uint64(h3.str_to_int(nbr_str))
        if nbr_id in h3_id_to_idx:
            j = h3_id_to_idx[nbr_id]
            dist_m, az_rad = haversine_dist_and_azimuth(lats[i], lons[i], lats[j], lons[j])
            
            neighbor_indices[i, nbr_count] = j
            neighbor_distances[i, nbr_count] = dist_m
            neighbor_azimuths[i, nbr_count] = az_rad
            nbr_count += 1
            
    neighbor_counts[i] = nbr_count

print(f"Topology built: mean {np.mean(neighbor_counts):.2f} neighbors per cell. Mean distance: {neighbor_distances[neighbor_distances > 0].mean():.1f} meters.")

# 3. Ingest Hourly ERA5 Weather Grid (24 hours)
weather_nc_path = os.path.join(RAW_DATA_DIR, "weather", "era5_hourly_aoi.nc")
ds_weather = xr.open_dataset(weather_nc_path).sortby("latitude")

# Bilinear interpolation across all 24 time slices for each H3 cell
points_lat = xr.DataArray(lats, dims="points")
points_lon = xr.DataArray(lons, dims="points")

hourly_u10 = ds_weather["u10"].interp(latitude=points_lat, longitude=points_lon, method="linear").values # (24, 2526)
hourly_v10 = ds_weather["v10"].interp(latitude=points_lat, longitude=points_lon, method="linear").values # (24, 2526)
hourly_t2m = ds_weather["t2m"].interp(latitude=points_lat, longitude=points_lon, method="linear").values # (24, 2526)
hourly_d2m = ds_weather["d2m"].interp(latitude=points_lat, longitude=points_lon, method="linear").values # (24, 2526)

# Compute wind speed (m/s) and meteorological FROM-direction (rad)
hourly_wind_speed = np.sqrt(hourly_u10**2 + hourly_v10**2).astype(np.float32)
# Meteorological wind direction: angle from which wind is blowing
hourly_wind_dir = np.mod(np.arctan2(-hourly_u10, -hourly_v10) + 2 * np.pi, 2 * np.pi).astype(np.float32)

# Estimate fuel moisture content from hourly RH and Temp
hourly_tc = hourly_t2m - 273.15
hourly_td = hourly_d2m - 273.15
hourly_rh = 100.0 * np.exp(17.625 * hourly_td / (243.04 + hourly_td)) / np.exp(17.625 * hourly_tc / (243.04 + hourly_tc))
# Simard (1968) Equilibrium Moisture Content approximation (fraction)
hourly_emc = np.clip(0.03 + 0.0025 * hourly_rh, 0.04, 0.20).astype(np.float32)

print(f"Hourly weather prepared: {hourly_wind_speed.shape[0]} hours across {hourly_wind_speed.shape[1]} cells.")

# 4. Seed Ignition Cells from MODIS and WFIGS
wfigs_path = os.path.join(RAW_DATA_DIR, "wfigs", "wfigs_perimeters.geojson")
gdf_wfigs = gpd.read_file(wfigs_path)

gdf_h3 = gpd.GeoDataFrame(
    df_static[["h3_id"]],
    geometry=[Point(lo, la) for lo, la in zip(lons, lats)],
    crs="EPSG:4326"
)

# Cells inside WFIGS initial discovery footprint
joined_wfigs = gpd.sjoin(gdf_h3, gdf_wfigs, how="inner", predicate="intersects")
ignition_h3_ids = set(joined_wfigs["h3_id"].unique())

# Also include MODIS detections on 2023-08-15
modis_path = os.path.join(RAW_DATA_DIR, "modis", "modis_2023_usa.csv")
df_modis = pd.read_csv(modis_path)
df_modis_aoi = df_modis[
    (df_modis["longitude"] >= -121.8) & (df_modis["longitude"] <= -121.3) &
    (df_modis["latitude"] >= 39.6) & (df_modis["latitude"] <= 40.0) &
    (df_modis["confidence"] >= 50)
]

for _, r in df_modis_aoi.iterrows():
    h_str = h3.latlng_to_cell(r["latitude"], r["longitude"], 8)
    h_u64 = np.uint64(h3.str_to_int(h_str))
    if h_u64 in h3_id_to_idx:
        ignition_h3_ids.add(h_u64)

ignition_indices = np.array([h3_id_to_idx[h_id] for h_id in ignition_h3_ids], dtype=np.int32)

print(f"\n--- Ignition Seeding Summary ---")
print(f"Total initial ignition cells: {len(ignition_indices)} / {num_cells} ({len(ignition_indices)/num_cells*100:.2f} % of AOI)")

Loaded H3 static features: 2526 cells.
Topology built: mean 5.83 neighbors per cell. Mean distance: 932.6 meters.
Hourly weather prepared: 24 hours across 2526 cells.

--- Ignition Seeding Summary ---
Total initial ignition cells: 337 / 2526 (13.34 % of AOI)


In [8]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Initialize Simulation State Arrays
INF_TIME = 1e9
arrival_times_min = np.full(num_cells, INF_TIME, dtype=np.float32)

# Set ignition cells at t = 0
for idx in ignition_indices:
    arrival_times_min[idx] = 0.0

dt_min = 5.0              # Simulation time step: 5 minutes
total_hours = 12          # 12-hour forecast horizon
total_steps = int((total_hours * 60) / dt_min)

print(f"Running Rothermel CA Simulation across {total_steps} steps (Horizon: {total_hours}h, dt: {dt_min} min)...")

# Active burning front tracking
currently_burned = np.zeros(num_cells, dtype=bool)
currently_burned[ignition_indices] = True

# 2. CA Simulation Loop
for step in range(total_steps):
    current_time_min = step * dt_min
    current_hour_idx = min(int(current_time_min // 60), 23)
    
    # Extract weather snapshot for the active hour
    ws_snap = hourly_wind_speed[current_hour_idx]
    wd_snap = hourly_wind_dir[current_hour_idx]
    moist_snap = hourly_emc[current_hour_idx]
    
    # Identify cells that are currently burning (arrival_time <= current_time)
    burning_cells = np.where(arrival_times_min <= current_time_min)[0]
    
    # Buffer to hold candidate arrival times for this time step
    candidate_updates = []
    
    for i in burning_cells:
        f_model = int(fuel_models[i])
        slp = float(slopes_rad[i])
        asp = float(aspects_rad[i])
        ws = float(ws_snap[i])
        wd = float(wd_snap[i])
        fm = float(moist_snap[i])
        
        n_count = neighbor_counts[i]
        for n_idx in range(n_count):
            j = neighbor_indices[i, n_idx]
            if j < 0:
                continue
                
            # If neighbor already burned earlier than current time, skip
            if arrival_times_min[j] <= current_time_min:
                continue
                
            az = float(neighbor_azimuths[i, n_idx])
            dist = float(neighbor_distances[i, n_idx])
            
            # Directional Rate of Spread from cell i to neighbor j (m/min)
            ros = rothermel_forward_ros(
                fuel_model=f_model,
                slope_rad=slp,
                aspect_rad=asp,
                wind_speed_ms=ws,
                wind_dir_rad=wd,
                neighbor_azimuth_rad=az,
                fuel_moisture=fm
            )
            
            if ros > 0.001:
                travel_time_min = dist / ros
                candidate_arrival = current_time_min + travel_time_min
                if candidate_arrival < arrival_times_min[j]:
                    candidate_updates.append((j, candidate_arrival))
                    
    # Atomic merge of candidate arrival times (serial minimum update)
    for j, cand_time in candidate_updates:
        if cand_time < arrival_times_min[j]:
            arrival_times_min[j] = cand_time

print("Simulation loop finished.")

# 3. Validation Metrics against Ground Truth
# Evaluate burned status at 3h, 6h, 12h
burned_at_3h = arrival_times_min <= (3 * 60)
burned_at_6h = arrival_times_min <= (6 * 60)
burned_at_12h = arrival_times_min <= (12 * 60)

# Ground truth burn mask from WFIGS perimeter
ground_truth_burned = np.zeros(num_cells, dtype=bool)
for idx in ignition_indices:
    ground_truth_burned[idx] = True

# IoU, Precision, Recall at 12h
intersection = np.logical_and(burned_at_12h, ground_truth_burned).sum()
union = np.logical_or(burned_at_12h, ground_truth_burned).sum()
iou_12h = intersection / union if union > 0 else 0.0
precision_12h = intersection / burned_at_12h.sum() if burned_at_12h.sum() > 0 else 0.0
recall_12h = intersection / ground_truth_burned.sum() if ground_truth_burned.sum() > 0 else 0.0

print("\n--- Rothermel CA 12-Hour Evaluation Metrics ---")
print(f"Ignition footprint:      {len(ignition_indices)} cells")
print(f"Predicted burned at 3h:  {burned_at_3h.sum()} cells ({burned_at_3h.sum()/num_cells*100:.1f} % of AOI)")
print(f"Predicted burned at 6h:  {burned_at_6h.sum()} cells ({burned_at_6h.sum()/num_cells*100:.1f} % of AOI)")
print(f"Predicted burned at 12h: {burned_at_12h.sum()} cells ({burned_at_12h.sum()/num_cells*100:.1f} % of AOI)")
print(f"12h Perimeter IoU:       {iou_12h:.4f}")
print(f"12h Precision:           {precision_12h:.4f}")
print(f"12h Recall:              {recall_12h:.4f}")

# 4. Parquet Output Assembly
df_ca_output = pd.DataFrame({
    "h3_id": df_static["h3_id"],
    "centroid_lon": df_static["centroid_lon"],
    "centroid_lat": df_static["centroid_lat"],
    "predicted_arrival_min": arrival_times_min,
    "is_burned_3h": burned_at_3h,
    "is_burned_6h": burned_at_6h,
    "is_burned_12h": burned_at_12h,
    "burn_time_step": np.where(arrival_times_min < INF_TIME, (arrival_times_min // dt_min).astype(np.int16), np.int16(-1))
})

ca_parquet_path = os.path.join(PROCESSED_DATA_DIR, "ca_simulation_camp_fire.parquet")
table = pa.Table.from_pandas(df_ca_output)
pq.write_table(table, ca_parquet_path, compression="snappy")

file_size_kb = os.path.getsize(ca_parquet_path) / 1024.0

print("\n=======================================================")
print(f"SUCCESS: Exported CA Simulation Parquet Table")
print(f"Path: {ca_parquet_path}")
print(f"Rows: {len(df_ca_output)}, File Size: {file_size_kb:.2f} KB")
print("=======================================================")
print(df_ca_output[df_ca_output["is_burned_12h"]].head(5))

Running Rothermel CA Simulation across 144 steps (Horizon: 12h, dt: 5.0 min)...
Simulation loop finished.

--- Rothermel CA 12-Hour Evaluation Metrics ---
Ignition footprint:      337 cells
Predicted burned at 3h:  852 cells (33.7 % of AOI)
Predicted burned at 6h:  1337 cells (52.9 % of AOI)
Predicted burned at 12h: 1981 cells (78.4 % of AOI)
12h Perimeter IoU:       0.1701
12h Precision:           0.1701
12h Recall:              1.0000

SUCCESS: Exported CA Simulation Parquet Table
Path: C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\ca_simulation_camp_fire.parquet
Rows: 2526, File Size: 58.31 KB
                  h3_id  centroid_lon  centroid_lat  predicted_arrival_min  \
166  613194646641704959   -121.406700     39.602604              92.116898   
167  613194646645899263   -121.396027     39.604939              51.338047   
168  613194646723493887   -121.450516     39.615295             128.963516   
169  613194646725591039   -121.458008     39.621082             138.916